# Gaussian Splatting — PLY Vergleich
**3mio.ply MCMC Postshot vs 3mio.ply Splat3 Postshot — PSNR, SSIM, LPIPS + Bildtafeln**

### Anleitung:
1. Oben: `Laufzeit → Laufzeittyp ändern → T4 GPU` aktivieren
2. Zellen von oben nach unten ausführen (▶)
3. Bei Schritt 3 die PLY-Dateien + Bilder hochladen

Pakete Installieren

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','gsplat','plyfile','lpips','-q'], check=True)
print('✅ Pakete installiert')

✅ Pakete installiert


Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive verbunden')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive verbunden


Konfig und Imports

In [ ]:
import os, gc, csv, warnings, statistics
from pathlib import Path
import numpy as np
import torch
from PIL import Image as PILImage
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import lpips as lp
import plyfile
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

DRIVE       = '/content/drive/MyDrive/GS_Analyse'
PLY_A       = f'{DRIVE}/3mio_MCMC_Postshot.ply'
PLY_B       = f'{DRIVE}/3mio_Splat3_Postshot.ply'
CAMERAS_TXT = f'{DRIVE}/cameras.txt'
IMAGES_TXT  = f'{DRIVE}/images.txt'
BILDER      = [
    f'{DRIVE}/Freiflug_DJI_0176_147.jpg',
    f'{DRIVE}/Freiflug_DJI_0315_286.jpg',
    f'{DRIVE}/Freiflug_DJI_0326_297.jpg',
    f'{DRIVE}/West_2_DJI_0362_2361.jpg',
    f'{DRIVE}/Nadir_West_DJI_0782_538.jpg',
]
OUT = f'{DRIVE}/vergleich'
os.makedirs(OUT, exist_ok=True)

device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lpips_device = torch.device('cpu')
lpips_fn     = lp.LPIPS(net='vgg', verbose=False).eval().to(lpips_device)

print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KEINE"}')
print(f'✅ LPIPS auf CPU | Ergebnisse → {OUT}')
for f in [PLY_A, PLY_B, CAMERAS_TXT, IMAGES_TXT] + BILDER:
    ok = Path(f).exists()
    sz = f'{Path(f).stat().st_size/1024/1024:.0f} MB' if ok else 'FEHLT'
    print(f'  {"✅" if ok else "❌"} {Path(f).name:<45} {sz}')

✅ GPU: Tesla T4
✅ LPIPS auf CPU | Ergebnisse → /content/drive/MyDrive/GS_Analyse/vergleich
  ✅ 3mio_MCMC_Postshot.ply                        675 MB
  ✅ 3mio_Splat3_Postshot.ply                      444 MB
  ✅ cameras.txt                                   0 MB
  ✅ images.txt                                    271 MB
  ✅ Freiflug_DJI_0176_147.jpg                     8 MB
  ✅ Freiflug_DJI_0315_286.jpg                     6 MB
  ✅ Freiflug_DJI_0326_297.jpg                     7 MB
  ✅ West_2_DJI_0362_2361.jpg                      6 MB
  ✅ Nadir_West_DJI_0782_538.jpg                   7 MB


COLMAP und Hilfsfunktionen

In [ ]:
from gsplat import rasterization
import math

MODEL_ID = {'SIMPLE_PINHOLE':0,'PINHOLE':1,'SIMPLE_RADIAL':2,'RADIAL':3,
            'OPENCV':4,'OPENCV_FISHEYE':5,'FULL_OPENCV':6,'FOV':7,
            'SIMPLE_RADIAL_FISHEYE':8,'RADIAL_FISHEYE':9,'THIN_PRISM_FISHEYE':10}

def read_cameras_txt(path):
    cams = {}
    with open(path) as f:
        for line in f:
            l = line.strip()
            if not l or l.startswith('#'): continue
            p = l.split()
            cams[int(p[0])] = {'model':MODEL_ID.get(p[1],1),'w':int(p[2]),'h':int(p[3]),
                               'params':[float(x) for x in p[4:]]}
    return cams

def read_images_txt(path):
    imgs = {}
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip() and not l.strip().startswith('#')]
    i = 0
    while i < len(lines):
        p = lines[i].split()
        imgs[p[9]] = {
            'qvec': np.array([float(p[1]),float(p[2]),float(p[3]),float(p[4])]),
            'tvec': np.array([float(p[5]),float(p[6]),float(p[7])]),
            'cam_id': int(p[8])
        }
        i += 2
    return imgs

def qvec_to_rotmat(q):
    qw,qx,qy,qz = q
    return np.array([[1-2*(qy**2+qz**2),2*(qx*qy-qw*qz),  2*(qx*qz+qw*qy)],
                     [2*(qx*qy+qw*qz),  1-2*(qx**2+qz**2),2*(qy*qz-qw*qx)],
                     [2*(qx*qz-qw*qy),  2*(qy*qz+qw*qx),  1-2*(qx**2+qy**2)]])

def render_one(ply_path, bild_pfad, cameras, images):
    bildname  = Path(bild_pfad).name
    ply_tag   = Path(ply_path).stem
    stem      = Path(bildname).stem
    save_path = f'{OUT}/{stem}_render_{ply_tag}.png'

    if Path(save_path).exists():
        print(f'  ⏭️  Bereits vorhanden: {Path(save_path).name} — übersprungen')
        return {'n':0,'save_path':save_path,'skipped':True}

    if bildname not in images:
        print(f'  ⚠️  {bildname} nicht in images.txt — SKIP'); return None

    # ── PLY laden ────────────────────────────────────────────
    print(f'  Lade {Path(ply_path).name}...', end=' ', flush=True)
    data = plyfile.PlyData.read(ply_path)
    v    = data['vertex']
    n    = len(v.data)

    xyz   = np.stack([v['x'],v['y'],v['z']],-1).astype(np.float32)
    sc    = np.stack([v['scale_0'],v['scale_1'],v['scale_2']],-1).astype(np.float32)
    qr    = np.stack([v['rot_0'],v['rot_1'],v['rot_2'],v['rot_3']],-1).astype(np.float32)
    op    = v['opacity'].astype(np.float32)
    sh_dc = np.stack([v['f_dc_0'],v['f_dc_1'],v['f_dc_2']], axis=-1).astype(np.float32)

    del data, v; gc.collect()

    # Rohe SH-Koeffizienten — gsplat macht C0*dc+0.5 intern selbst
    sh_all         = sh_dc[:,None,:].astype(np.float32)   # (N,1,3) — keine Vorverarbeitung
    sh_degree_used = 0

    means  = torch.tensor(xyz)
    scales = torch.tensor(np.exp(sc))
    quats  = torch.tensor(qr)
    quats  = quats / (quats.norm(dim=-1, keepdim=True) + 1e-8)
    opacs  = torch.tensor(1. / (1. + np.exp(-op)))
    colors = torch.tensor(sh_all)   # (N,1,3)
    del xyz, sc, qr, op, sh_dc, sh_all; gc.collect()
    print(f'{n/1e6:.2f}M Splats ✅')

    # ── Kameraparameter ───────────────────────────────────────
    entry  = images[bildname]
    cam    = cameras[entry['cam_id']]
    W, H   = cam['w'], cam['h']
    p      = cam['params']
    fx, fy = (p[0], p[0]) if cam['model'] == 0 else (p[0], p[1])
    cx, cy = (p[1], p[2]) if cam['model'] == 0 else (p[2], p[3])
    R, T   = qvec_to_rotmat(entry['qvec']), entry['tvec']

    # ── Rendern ───────────────────────────────────────────────
    print('  Rendere...', end=' ', flush=True)
    Rt = np.eye(4, dtype=np.float32)
    Rt[:3,:3] = R; Rt[:3,3] = T
    vm = torch.tensor(Rt).unsqueeze(0).to(device)
    K  = torch.tensor([[fx,0,cx],[0,fy,cy],[0,0,1]], dtype=torch.float32).unsqueeze(0).to(device)

    renders, _, _ = rasterization(
        means      = means.to(device),
        quats      = quats.to(device),
        scales     = scales.to(device),
        opacities  = opacs.to(device),
        colors     = colors.to(device),
        viewmats   = vm,
        Ks         = K,
        width      = W,
        height     = H,
        sh_degree  = sh_degree_used,
        near_plane = 0.01,
        far_plane  = 1000.0,
    )
    img = renders[0].clamp(0, 1).cpu().numpy()
    print('✅')

    # ── Sofort in Drive speichern ─────────────────────────────
    PILImage.fromarray((img * 255).astype(np.uint8)).save(save_path)
    print(f'  💾 Gespeichert: {Path(save_path).name}')

    # ── RAM + VRAM leeren ─────────────────────────────────────
    del means, scales, quats, opacs, colors, renders, vm, K, img
    gc.collect(); torch.cuda.empty_cache()
    print('  🗑️  RAM geleert\n')
    return {'n': n, 'save_path': save_path, 'skipped': False}

cameras = read_cameras_txt(CAMERAS_TXT)
images  = read_images_txt(IMAGES_TXT)
print(f'✅ COLMAP: {len(cameras)} Kamera(s), {len(images)} Bilder')
print('✅ Bereit — alte Renders im Drive-Ordner vergleich/ zuerst löschen!')

✅ COLMAP: 1 Kamera(s), 2762 Bilder
✅ Bereit — alte Renders im Drive-Ordner vergleich/ zuerst löschen!


Testing

In [ ]:
import plyfile
import numpy as np

data  = plyfile.PlyData.read(PLY_A)
v     = data['vertex']
sh_dc = np.stack([v['f_dc_0'], v['f_dc_1'], v['f_dc_2']], axis=-1).astype(np.float32)

print(f'DC min:  {sh_dc.min():.4f}')
print(f'DC max:  {sh_dc.max():.4f}')
print(f'DC mean: {sh_dc.mean():.4f}')
print(f'DC std:  {sh_dc.std():.4f}')
print(f'Erste 5 Zeilen:\n{sh_dc[:5]}')
del data, v

DC min:  -3.6291
DC max:  11.6518
DC mean: 0.0266
DC std:  1.1532
Erste 5 Zeilen:
[[-0.21789153 -0.06676614  0.13418603]
 [ 0.10884228  0.26569265  0.3172251 ]
 [-0.770971   -0.5741898  -0.42699677]
 [ 0.4377935   0.5959524   0.71843004]
 [-0.40654317 -0.24753149 -0.1024979 ]]


Render 3 Mio MCMC Postshot

In [ ]:
print('=== 3mio_MCMC_Postshot.ply × Bild 1 ===')
r1a = render_one(PLY_A, BILDER[0], cameras, images)

=== 3mio_MCMC_Postshot.ply × Bild 1 ===
  Lade 3mio_MCMC_Postshot.ply... 3.00M Splats ✅
  Rendere... ✅
  💾 Gespeichert: Freiflug_DJI_0176_147_render_3mio_MCMC_Postshot.png
  🗑️  RAM geleert



In [ ]:
print('=== 3mio_MCMC_Postshot.ply × Bild 2 ===')
r2a = render_one(PLY_A, BILDER[1], cameras, images)

=== 3mio_MCMC_Postshot.ply × Bild 2 ===
  Lade 3mio_MCMC_Postshot.ply... 3.00M Splats ✅
  Rendere... ✅
  💾 Gespeichert: Freiflug_DJI_0315_286_render_3mio_MCMC_Postshot.png
  🗑️  RAM geleert



In [ ]:
print('=== 3mio_MCMC_Postshot.ply × Bild 3 ===')
r3a = render_one(PLY_A, BILDER[2], cameras, images)

=== 3mio_MCMC_Postshot.ply × Bild 3 ===
  Lade 3mio_MCMC_Postshot.ply... 3.00M Splats ✅
  Rendere... ✅
  💾 Gespeichert: Freiflug_DJI_0326_297_render_3mio_MCMC_Postshot.png
  🗑️  RAM geleert



In [ ]:
print('=== 3mio_MCMC_Postshot.ply × Bild 4 ===')
r4a = render_one(PLY_A, BILDER[3], cameras, images)

=== 3mio_MCMC_Postshot.ply × Bild 4 ===
  Lade 3mio_MCMC_Postshot.ply... 3.00M Splats ✅
  Rendere... ✅
  💾 Gespeichert: West_2_DJI_0362_2361_render_3mio_MCMC_Postshot.png
  🗑️  RAM geleert



In [ ]:
print('=== 3mio_MCMC_Postshot.ply × Bild 5 ===')
r5a = render_one(PLY_A, BILDER[4], cameras, images)

=== 3mio_MCMC_Postshot.ply × Bild 5 ===
  Lade 3mio_MCMC_Postshot.ply... 3.00M Splats ✅
  Rendere... ✅
  💾 Gespeichert: Nadir_West_DJI_0782_538_render_3mio_MCMC_Postshot.png
  🗑️  RAM geleert



Render 3 Mio Splat3 Postshot

In [ ]:
print('=== 3mio_Splat3_Postshot.ply × Bild 1 ===')
r1b = render_one(PLY_B, BILDER[0], cameras, images)

=== 3mio_Splat3_Postshot.ply × Bild 1 ===
  Lade 3mio_Splat3_Postshot.ply... 3.07M Splats ✅
  Rendere... ✅
  💾 Gespeichert: Freiflug_DJI_0176_147_render_3mio_Splat3_Postshot.png
  🗑️  RAM geleert



In [ ]:
print('=== 3mio_Splat3_Postshot.ply × Bild 2 ===')
r2b = render_one(PLY_B, BILDER[1], cameras, images)

=== 3mio_Splat3_Postshot.ply × Bild 2 ===
  Lade 3mio_Splat3_Postshot.ply... 3.07M Splats ✅
  Rendere... ✅
  💾 Gespeichert: Freiflug_DJI_0315_286_render_3mio_Splat3_Postshot.png
  🗑️  RAM geleert



In [ ]:
print('=== 3mio_Splat3_Postshot.ply × Bild 3 ===')
r3b = render_one(PLY_B, BILDER[2], cameras, images)

=== 3mio_Splat3_Postshot.ply × Bild 3 ===
  Lade 3mio_Splat3_Postshot.ply... 3.07M Splats ✅
  Rendere... ✅
  💾 Gespeichert: Freiflug_DJI_0326_297_render_3mio_Splat3_Postshot.png
  🗑️  RAM geleert



In [ ]:
print('=== 3mio_Splat3_Postshot.ply × Bild 4 ===')
r4b = render_one(PLY_B, BILDER[3], cameras, images)

=== 3mio_Splat3_Postshot.ply × Bild 4 ===
  Lade 3mio_Splat3_Postshot.ply... 3.07M Splats ✅
  Rendere... ✅
  💾 Gespeichert: West_2_DJI_0362_2361_render_3mio_Splat3_Postshot.png
  🗑️  RAM geleert



In [ ]:
print('=== 3mio_Splat3_Postshot.ply × Bild 5 ===')
r5b = render_one(PLY_B, BILDER[4], cameras, images)

=== 3mio_Splat3_Postshot.ply × Bild 5 ===
  Lade 3mio_Splat3_Postshot.ply... 3.07M Splats ✅
  Rendere... ✅
  💾 Gespeichert: Nadir_West_DJI_0782_538_render_3mio_Splat3_Postshot.png
  🗑️  RAM geleert



Metriken + Bildtafeln

In [ ]:
all_results = []

for bild_pfad in BILDER:
    bildname = Path(bild_pfad).name
    stem = Path(bildname).stem
    print(f'\n── {bildname}')

    path_a = f'{OUT}/{stem}_render_3mio_MCMC_Postshot.png'
    path_b = f'{OUT}/{stem}_render_3mio_Splat3_Postshot.png'

    if not Path(path_a).exists() or not Path(path_b).exists():
        print(' ⚠️ Render fehlt — Zellen 5–10 zuerst ausführen!'); continue

    entry = images[bildname]; cam = cameras[entry['cam_id']]
    W,H = cam['w'],cam['h']
    orig = np.array(PILImage.open(bild_pfad).convert('RGB').resize((W,H),PILImage.LANCZOS),
                    dtype=np.float32)/255.
    ra = np.array(PILImage.open(path_a).convert('RGB'),dtype=np.float32)/255.
    rb = np.array(PILImage.open(path_b).convert('RGB'),dtype=np.float32)/255.

    print(' PSNR + SSIM...', end=' ', flush=True)
    psnr_a = float(peak_signal_noise_ratio(orig, ra, data_range=1.0))
    ssim_a = float(structural_similarity(orig, ra, data_range=1.0, channel_axis=2))
    psnr_b = float(peak_signal_noise_ratio(orig, rb, data_range=1.0))
    ssim_b = float(structural_similarity(orig, rb, data_range=1.0, channel_axis=2))
    print('✅')

    print(' LPIPS...', end=' ', flush=True)
    def to_lpips_tensor_small(arr):
        h, w = arr.shape[:2]
        scale = 512 / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img_small = np.array(
            PILImage.fromarray((arr * 255).astype(np.uint8)).resize((new_w, new_h), PILImage.LANCZOS),
            dtype=np.float32
        ) / 255.
        t = torch.tensor(img_small, dtype=torch.float32).permute(2,0,1).unsqueeze(0) * 2 - 1
        del img_small
        return t.cpu()
    t_orig = to_lpips_tensor_small(orig)
    t_ra   = to_lpips_tensor_small(ra)
    t_rb   = to_lpips_tensor_small(rb)
    with torch.no_grad():
        lpips_a = float(lpips_fn(t_ra, t_orig))
        lpips_b = float(lpips_fn(t_rb, t_orig))
    del t_orig, t_ra, t_rb
    gc.collect()
    print('✅')

    print(f'   PLY_A  → PSNR={psnr_a:.2f} dB  SSIM={ssim_a:.4f}  LPIPS={lpips_a:.4f}')
    print(f'   PLY_B → PSNR={psnr_b:.2f} dB  SSIM={ssim_b:.4f}  LPIPS={lpips_b:.4f}')

    PILImage.fromarray((orig*255).astype(np.uint8)).save(f'{OUT}/{stem}_original.png')
    del orig, ra, rb; gc.collect()

    imgs_plot = [
        (np.array(PILImage.open(f'{OUT}/{stem}_original.png').convert('RGB'))/255., 'Original',        None,   None,   None),
        (np.array(PILImage.open(path_a).convert('RGB'))/255.,                       '3 Mio. Splats Jawset Postshot mit "MCMC"',   psnr_a, ssim_a, lpips_a),
        (np.array(PILImage.open(path_b).convert('RGB'))/255.,                       '3 Mio. Splats Jawset Postshot mit "Splat3"',  psnr_b, ssim_b, lpips_b),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    fig.patch.set_facecolor('white')
    for ax,(img,title,ps,ss,lp) in zip(axes, imgs_plot):
        ax.imshow(img); ax.set_facecolor('white')
        ax.set_title(title, color='black', fontsize=18, fontweight='bold', pad=6)
        if ps is not None:
            ax.set_xlabel(f'PSNR {ps:.2f} dB  |  SSIM {ss:.4f}  |  LPIPS {lp:.4f}',
                          color='#2a6a2a', fontsize=15)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        for spine in ax.spines.values(): spine.set_visible(False)
    plt.suptitle(bildname, color='#555555', fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'{OUT}/{stem}_vergleich.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show(); plt.close()
    del imgs_plot; gc.collect()
    print(f'  💾 {stem}_vergleich.png gespeichert')

    all_results += [
        {'Bild':bildname,'PLY':'3mio_MCMC_Postshot.ply', 'PSNR_dB':round(psnr_a,4),'SSIM':round(ssim_a,6),'LPIPS':round(lpips_a,6)},
        {'Bild':bildname,'PLY':'3mio_Splat3_Postshot.ply','PSNR_dB':round(psnr_b,4),'SSIM':round(ssim_b,6),'LPIPS':round(lpips_b,6)},
    ]

print('\n✅ Alle Metriken fertig')


── Freiflug_DJI_0176_147.jpg
 PSNR + SSIM... ✅
 LPIPS... ✅
   PLY_A  → PSNR=12.63 dB  SSIM=0.4633  LPIPS=0.3433
   PLY_B → PSNR=12.47 dB  SSIM=0.4660  LPIPS=0.3115
  💾 Freiflug_DJI_0176_147_vergleich.png gespeichert

── Freiflug_DJI_0315_286.jpg
 PSNR + SSIM... ✅
 LPIPS... ✅
   PLY_A  → PSNR=16.32 dB  SSIM=0.4996  LPIPS=0.3278
   PLY_B → PSNR=16.68 dB  SSIM=0.5011  LPIPS=0.3182
  💾 Freiflug_DJI_0315_286_vergleich.png gespeichert

── Freiflug_DJI_0326_297.jpg
 PSNR + SSIM... ✅
 LPIPS... ✅
   PLY_A  → PSNR=15.88 dB  SSIM=0.5392  LPIPS=0.3228
   PLY_B → PSNR=16.62 dB  SSIM=0.5597  LPIPS=0.2768
  💾 Freiflug_DJI_0326_297_vergleich.png gespeichert

── West_2_DJI_0362_2361.jpg
 PSNR + SSIM... ✅
 LPIPS... ✅
   PLY_A  → PSNR=20.95 dB  SSIM=0.6826  LPIPS=0.1761
   PLY_B → PSNR=20.85 dB  SSIM=0.6842  LPIPS=0.1595
  💾 West_2_DJI_0362_2361_vergleich.png gespeichert

── Nadir_West_DJI_0782_538.jpg
 PSNR + SSIM... ✅
 LPIPS... ✅
   PLY_A  → PSNR=19.16 dB  SSIM=0.6301  LPIPS=0.2405
   PLY_B → PSNR=20.

Zusammenfassung & CSV

In [ ]:
def avg(key,ply): return statistics.mean([r[key] for r in all_results if r['PLY']==ply])

print(f'\n{"="*63}')
print(f'  Durchschnitt über {len(BILDER)} Bilder')
print(f'  {"─"*59}')
print(f'  {"Metrik":<8} {"3mio_MCMC_Postshot.ply":<28} 3mio_MCMC_Lichtfeld.ply')
print(f'  {"─"*59}')
print(f'  PSNR   {avg("PSNR_dB","3mio_MCMC_Postshot.ply"):>6.2f} dB              {avg("PSNR_dB","3mio_Splat3_Postshot.ply"):>6.2f} dB')
print(f'  SSIM   {avg("SSIM","3mio_MCMC_Postshot.ply"):>8.4f}              {avg("SSIM","3mio_Splat3_Postshot.ply"):>8.4f}')
print(f'  LPIPS  {avg("LPIPS","3mio_MCMC_Postshot.ply"):>8.4f}              {avg("LPIPS","3mio_Splat3_Postshot.ply"):>8.4f}')
print(f'{"="*63}')
print()
print('  PSNR  → höher ist besser  (gut: >25 dB)')
print('  SSIM  → höher ist besser  (gut: >0.80)')
print('  LPIPS → tiefer ist besser (gut: <0.20)')

csv_path = f'{OUT}/gs_metriken.csv'
with open(csv_path,'w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['Bild','PLY','PSNR_dB','SSIM','LPIPS'])
    w.writeheader()
    w.writerows(all_results)
    w.writerow({})
    for ply_name in ['3mio_MCMC_Postshot.ply','3mio_Splat3_Postshot.ply']:
        w.writerow({'Bild':'DURCHSCHNITT','PLY':ply_name,
                    'PSNR_dB': round(avg('PSNR_dB', ply_name),4),
                    'SSIM':    round(avg('SSIM',    ply_name),6),
                    'LPIPS':   round(avg('LPIPS',   ply_name),6)})

print(f'\n✅ CSV: {csv_path}')
print(f'✅ Alle Dateien in Drive: {OUT}/')


  Durchschnitt über 5 Bilder
  ───────────────────────────────────────────────────────────
  Metrik   3mio_MCMC_Postshot.ply       3mio_MCMC_Lichtfeld.ply
  ───────────────────────────────────────────────────────────
  PSNR    16.99 dB               17.33 dB
  SSIM     0.5629                0.5671
  LPIPS    0.2821                0.2540

  PSNR  → höher ist besser  (gut: >25 dB)
  SSIM  → höher ist besser  (gut: >0.80)
  LPIPS → tiefer ist besser (gut: <0.20)

✅ CSV: /content/drive/MyDrive/GS_Analyse/vergleich/gs_metriken.csv
✅ Alle Dateien in Drive: /content/drive/MyDrive/GS_Analyse/vergleich/
